# Legal RAG — Retrieval Evaluation (Kaggle)
Khong can LLM. Chi: embed query + dense + BM25 + fuse + rerank + extract articles + F2-Macro.

**Dataset:** `ai-rag-legal-assets`
- `corpus.zip`: corpus.jsonl (4.1 GB)
- `indexes.zip`: dense.index + BM25 (4.9 GB)
- `src.zip`: ma nguon project

In [ ]:
# Cell 1: Install deps
import subprocess, sys
deps = [
    'transformers>=4.48.0', 'accelerate>=1.3.0',
    'sentencepiece>=0.2.0', 'datasets>=2.19.0', 'faiss-gpu>=1.9.0',
    'bm25s>=0.3.0', 'sentence-transformers>=3.3.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + deps)
print('Deps installed')

In [ ]:
# Cell 2: Extract assets
import os, sys, zipfile, json, re, time
from pathlib import Path

DATASET = Path('/kaggle/input/ai-rag-legal-assets')
WORKING = Path('/kaggle/working')

for zname in ('corpus.zip', 'indexes.zip', 'src.zip'):
    with zipfile.ZipFile(DATASET / zname) as z:
        z.extractall(WORKING)
    print(f'Extracted {zname}')

os.environ['HF_HOME'] = str(WORKING / 'hf_cache')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
sys.path.insert(0, str(WORKING))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('Ready')

In [ ]:
# Cell 3: Load corpus + indexes
t0 = time.time()
from src.data.loading import load_corpus
docs = load_corpus(
    cache_path=str(WORKING / 'data/processed/corpus.jsonl'),
    force_rebuild=False
)
print(f'Corpus: {len(docs)} docs ({time.time()-t0:.1f}s)')

t0 = time.time()
from src.retrieval.indexing import SparseIndex, DenseIndex
si = SparseIndex(); si.load(str(WORKING / 'data/indexes/sparse'))
print(f'BM25: {len(si.doc_ids)} docs ({time.time()-t0:.1f}s)')

di = DenseIndex(); di.load(str(WORKING / 'data/indexes/dense.index'))
print(f'FAISS: {di.index.ntotal} vectors ({time.time()-t0:.1f}s)')

In [ ]:
# Cell 4: Load embedder + reranker
t0 = time.time()
from src.embedding.harrier_embedding import HarrierEmbedding
embedder = HarrierEmbedding(device='cuda')
print(f'Harrier: {time.time()-t0:.1f}s')

t0 = time.time()
from src.reranker.cross_encoder import CrossEncoderReranker
ce = CrossEncoderReranker(device='cuda')
print(f'Reranker: {time.time()-t0:.1f}s')

In [ ]:
# Cell 5: Retrieval logic (async, khong LLM)
import numpy as np
from src.core.base import RetrievedChunk
from src.retrieval.indexing import rrf_fusion

E5_PREFIX = 'Voi mot truy van ve luat Viet Nam, truy xuat cac doan van lien quan co chua cau tra loi cho truy van do'

async def retrieve(query: str, top_k: int = 500, dw: float = 0.8, sw: float = 0.2):
    embs = await embedder.embed([f'{E5_PREFIX}\nTruy van: {query}'])
    q_emb = np.array(embs[0])
    dense_results = di.search(q_emb, top_k)
    sparse_results = si.search(query, top_k)
    fused = rrf_fusion(dense_results, sparse_results, weights=[dw, sw], top_k=top_k)
    chunks = []
    for doc_id, score in fused:
        if doc_id < len(docs):
            d = docs[doc_id]
            chunks.append(RetrievedChunk(
                chunk_id=str(doc_id), doc_id=doc_id,
                article_id=d.get('article_id', ''),
                doc_title=d.get('source', ''),
                content=d.get('text', '')[:2000],
                score=score, retrieval_score=score, rerank_score=score,
                source='hybrid', metadata={},
            ))
    return chunks

async def extract_articles(chunks, top_k: int = 50):
    reranked = await ce.rerank('', chunks[:top_k], top_k)
    articles = []
    for c in reranked:
        if c.article_id and c.article_id not in articles:
            articles.append(c.article_id)
    return articles

print('Retrieval functions ready')

In [ ]:
# Cell 6: Load evaluation set (PBGDPL)
from datasets import load_dataset

ds = load_dataset('tmquan/pbgdpl-vn-legal-qna', split='train', streaming=True)
eval_set = []
for row in ds:
    q = row.get('question_text', row.get('question', ''))
    a = row.get('answer_text', row.get('answer', ''))
    if not q or not a:
        continue
    ref = list(set(re.findall(r'\u0110i\u1ec1u\s+(\d+)', str(a))))
    if len(ref) >= 2:
        eval_set.append({'question': q, 'reference_articles': ref})
    if len(eval_set) >= 200:
        break
print(f'Eval set: {len(eval_set)} queries')

In [ ]:
# Cell 7: Run evaluation
from src.evaluation.metrics import compute_f2_macro, compute_retrieval_metrics

all_preds = []
for i, item in enumerate(eval_set):
    q = item['question']
    gt = item['reference_articles']
    chunks = await retrieve(q)
    pred = await extract_articles(chunks)
    all_preds.append({'predicted': pred, 'ground_truth': gt, 'query': q})
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(eval_set)}')

y_true = [p['ground_truth'] for p in all_preds]
y_pred = [p['predicted'] for p in all_preds]
f2 = compute_f2_macro(y_true, y_pred)
ret = compute_retrieval_metrics(y_true, y_pred, k_values=[5, 10, 20, 50])
results = {**f2, **ret, 'num_queries': len(y_true)}

print(f'\n{"="*50}')
print(f'EVALUATION ({len(y_true)} queries)')
print(f'{"="*50}')
print(f'Macro-F2:  {results["macro_f2"]:.4f}')
print(f'Micro-F2:  {results["micro_f2"]:.4f}')
print(f'Micro-Recall: {results["micro_recall"]:.4f}')
for k in [5, 10, 20, 50]:
    print(f'Recall@{k}: {results.get(f"recall@{k}", 0):.4f}')
print(f'{"="*50}')

with open(WORKING / 'eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(WORKING / 'eval_detail.json', 'w', encoding='utf-8') as f:
    json.dump(all_preds, f, ensure_ascii=False, indent=2, default=str)
print(f'Saved')